# Day 4 · 08 · Performance Tuning

Performance work has one rule: **measure first**. Then spend effort where the time and memory actually go.

**In this notebook**
- Measuring memory and time
- Smaller dtypes: integers, floats and `category`
- What `category` does for `groupby`
- Reading less: `usecols`, `dtype`, `chunksize`
- CSV versus Parquet
- Repeated lookups: index instead of mask
- Building a frame: concatenate once
- A tuning checklist

**Dataset:** the retail order lines, repeated 50 times to get about 220,000 rows. Temporary files are written to a temporary folder and removed at the end.

In [1]:
import shutil
import tempfile
import timeit
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")

base = (
    items.merge(orders, on="order_id")
    .merge(products[["product_id", "category"]], on="product_id")
    .merge(customers[["customer_id", "country", "segment"]], on="customer_id")
)

COPIES = 50
big = pd.concat([base] * COPIES, ignore_index=True)
big["order_id"] += np.repeat(np.arange(COPIES) * 10_000, len(base))   # keep order ids distinct per copy
big["revenue"] = big["quantity"] * big["unit_price"] * (1 - big["discount"])


def best_ms(function, repeat=3):
    """Best of `repeat` runs, in milliseconds."""
    return min(timeit.repeat(function, number=1, repeat=repeat)) * 1000


def megabytes(frame):
    return frame.memory_usage(deep=True).sum() / 1e6


print(f"{len(big):,} rows x {big.shape[1]} columns, {megabytes(big):.1f} MB")

221,950 rows x 15 columns, 36.4 MB


## 1. Measure memory
- `memory_usage(deep=True)` reports bytes per column. `deep=True` includes the text itself, not just pointers to it.
- `df.info(memory_usage="deep")` prints the same total with the dtypes.

Text columns usually dominate.

In [2]:
usage = pd.DataFrame(
    {
        "dtype": big.dtypes.astype("str"),
        "distinct": big.nunique(),
        "megabytes": big.memory_usage(deep=True).drop("Index") / 1e6,
    }
)
usage.sort_values("megabytes", ascending=False).round(2)

,dtype,distinct,megabytes
segment,str,3,3.79
status,str,4,3.75
category,str,5,3.16
country,str,8,2.94
customer_id,str,205,2.91
channel,str,3,2.77
product_id,str,37,2.66
staff_id,str,8,1.94
discount,float64,5,1.78
quantity,int64,5,1.78


## 2. Choose smaller dtypes
| Column holds | Default | Smaller option |
| --- | --- | --- |
| small whole numbers | `int64` (8 bytes) | `int8`, `int16`, `int32` via `pd.to_numeric(..., downcast="integer")` |
| decimals | `float64` | `float32`, if 7 significant digits are enough |
| text with few distinct values | `str` | `category` |

A `category` column stores each distinct text once, plus a small integer code per row. It suits columns such as status or country. It does **not** help when almost every value is different, such as names or free text.

In [3]:
slim = big.copy()

for column in ["product_id", "customer_id", "status", "channel", "staff_id", "category", "country", "segment"]:
    slim[column] = slim[column].astype("category")

for column in ["order_id", "line_no", "quantity"]:
    slim[column] = pd.to_numeric(slim[column], downcast="integer")

for column in ["unit_price", "discount"]:
    slim[column] = slim[column].astype("float32")

comparison = pd.DataFrame(
    {
        "before": big.dtypes.astype("str"),
        "after": slim.dtypes.astype("str"),
        "mb_before": big.memory_usage(deep=True).drop("Index") / 1e6,
        "mb_after": slim.memory_usage(deep=True).drop("Index") / 1e6,
    }
).round(2)

print(f"total: {megabytes(big):.1f} MB -> {megabytes(slim):.1f} MB ({megabytes(slim) / megabytes(big):.0%} of the original)")
comparison

total: 36.4 MB -> 8.7 MB (24% of the original)


,before,after,mb_before,mb_after
order_id,int64,int32,1.78,0.89
line_no,int64,int8,1.78,0.22
product_id,str,category,2.66,0.22
quantity,int64,int8,1.78,0.22
unit_price,float64,float32,1.78,0.89
discount,float64,float32,1.78,0.89
customer_id,str,category,2.91,0.45
order_date,datetime64[us],datetime64[us],1.78,1.78
status,str,category,3.75,0.22
channel,str,category,2.77,0.22


### The price of `float32`
Smaller floats are less precise. Totals drift slightly, which is fine for charts and unacceptable for accounting. `revenue` was left as `float64` for that reason.

In [4]:
exact = (big["quantity"] * big["unit_price"]).sum()
approximate = (slim["quantity"] * slim["unit_price"]).sum()

print(f"float64 total: {exact:,.2f}")
print(f"float32 total: {approximate:,.2f}")
print(f"difference   : {abs(exact - approximate):,.2f}")

float64 total: 21,191,087.00
float32 total: 21,191,088.00
difference   : 1.00


## 3. `category` and `groupby`
Grouping by text means pandas must first work out the distinct values. A `category` column already knows them.

Pass `observed=True` when grouping by categories so that only combinations present in the data are returned.

In [5]:
def group_text():
    return big.groupby(["country", "category"])["revenue"].sum()


def group_category():
    return slim.groupby(["country", "category"], observed=True)["revenue"].sum()


print("same result:", np.allclose(group_text().to_numpy(), group_category().to_numpy()))
print(f"text keys     : {best_ms(group_text, repeat=5):7.2f} ms")
print(f"category keys : {best_ms(group_category, repeat=5):7.2f} ms")

same result: True
text keys     :   20.78 ms
category keys :   11.68 ms


## 4. Load less
Data you never load costs no memory. `read_csv` can skip columns, set dtypes while parsing, and stream a file in pieces.

| Argument | Effect |
| --- | --- |
| `usecols=[...]` | parse only these columns |
| `dtype={...}` | set types during the read instead of converting afterwards |
| `parse_dates=[...]` | parse date columns |
| `nrows=1000` | read a sample |
| `chunksize=50_000` | return an iterator of DataFrames |

In [6]:
workdir = Path(tempfile.mkdtemp(prefix="perf_demo_"))
csv_path = workdir / "sales.csv"
big.to_csv(csv_path, index=False)

needed = ["order_date", "category", "quantity", "unit_price", "discount"]
typed = {"category": "category", "quantity": "int16"}


def read_all():
    return pd.read_csv(csv_path)


def read_needed():
    return pd.read_csv(csv_path, usecols=needed)


def read_needed_typed():
    return pd.read_csv(csv_path, usecols=needed, dtype=typed, parse_dates=["order_date"])


pd.DataFrame(
    {
        "milliseconds": {
            "all columns": best_ms(read_all),
            "usecols": best_ms(read_needed),
            "usecols + dtype + parse_dates": best_ms(read_needed_typed),
        },
        "megabytes": {
            "all columns": megabytes(read_all()),
            "usecols": megabytes(read_needed()),
            "usecols + dtype + parse_dates": megabytes(read_needed_typed()),
        },
    }
).round(1)

,milliseconds,megabytes
all columns,40.4,38.4
usecols,74.1,12.4
usecols + dtype + parse_dates,85.6,6.0


Read the two columns separately:
- **Memory** falls sharply with each step. That is the reliable gain.
- **Time** did not improve in this run: reading every column was the fastest. With this pandas version and file, selecting columns and parsing dates cost more than they saved.

So treat `usecols` and `dtype` first of all as memory tools, and time the read on your own files before assuming it got faster.

### Files larger than memory: `chunksize`
With `chunksize`, `read_csv` yields one DataFrame per block of rows. Aggregate each block, keep only the small result, and combine at the end. Memory use stays at one chunk, whatever the file size.

In [7]:
partials = []
chunks = 0
for chunk in pd.read_csv(csv_path, usecols=["category", "quantity", "unit_price", "discount"], chunksize=50_000):
    chunk_revenue = chunk["quantity"] * chunk["unit_price"] * (1 - chunk["discount"])
    partials.append(chunk_revenue.groupby(chunk["category"]).sum())
    chunks += 1

from_chunks = pd.concat(partials).groupby(level=0).sum()
in_memory = big.groupby("category")["revenue"].sum()

print("chunks read:", chunks)
print("matches the in-memory result:", np.allclose(from_chunks.sort_index(), in_memory.sort_index()))
from_chunks.round(0)

chunks read: 5
matches the in-memory result: True


category
Books          3831562.0
Electronics    7921510.0
Home           2912291.0
Sports         3572866.0
Toys           2020889.0
dtype: float64

## 5. CSV versus Parquet
CSV is text: every read has to parse every character again, and the dtypes are lost. **Parquet** is a binary, column-oriented format that:
- stores dtypes, so dates stay dates and categories stay categories
- compresses each column
- can read only the columns you ask for

Reading and writing Parquet needs the `pyarrow` package. The Parquet file below is written from the slimmed frame, so its smaller dtypes travel with it.

In [8]:
parquet_path = workdir / "sales.parquet"
slim.to_parquet(parquet_path, index=False)

formats = pd.DataFrame(
    {
        "file_mb": {"CSV": csv_path.stat().st_size / 1e6, "Parquet": parquet_path.stat().st_size / 1e6},
        "read_all_ms": {"CSV": best_ms(read_all), "Parquet": best_ms(lambda: pd.read_parquet(parquet_path))},
        "read_3_columns_ms": {
            "CSV": best_ms(lambda: pd.read_csv(csv_path, usecols=["category", "quantity", "revenue"])),
            "Parquet": best_ms(lambda: pd.read_parquet(parquet_path, columns=["category", "quantity", "revenue"])),
        },
    }
).round(1)
formats

,file_mb,read_all_ms,read_3_columns_ms
CSV,20.0,48.3,56.7
Parquet,1.9,15.5,6.7


In [9]:
# Parquet round-trips the dtypes. CSV returns dates and categories as plain text.
check = ["order_date", "status", "quantity"]
pd.DataFrame(
    {
        "written": slim[check].dtypes.astype("str"),
        "from_parquet": pd.read_parquet(parquet_path, columns=check).dtypes.astype("str"),
        "from_csv": pd.read_csv(csv_path, usecols=check).dtypes.astype("str"),
    }
)

,written,from_parquet,from_csv
order_date,datetime64[us],datetime64[us],str
quantity,int8,int8,int64
status,category,category,str


## 6. Repeated lookups: use the index
`df[df["key"] == value]` scans the whole column every time. For many lookups by key:
- set the key as a **sorted index** and use `.loc`, or
- better, do all the lookups at once with `isin`, a merge or a `groupby`.

In [10]:
keys = big["order_id"].drop_duplicates().sample(200, random_state=1).tolist()
indexed = big.set_index("order_id").sort_index()


def lookup_by_mask():
    return [big.loc[big["order_id"] == key, "revenue"].sum() for key in keys]


def lookup_by_index():
    return [indexed.loc[[key], "revenue"].sum() for key in keys]


def lookup_all_at_once():
    return big[big["order_id"].isin(keys)].groupby("order_id")["revenue"].sum().loc[keys].tolist()


print("same result:", np.allclose(lookup_by_mask(), lookup_by_index()), np.allclose(lookup_by_mask(), lookup_all_at_once()))
pd.Series(
    {
        "200 boolean-mask scans": best_ms(lookup_by_mask),
        "200 .loc lookups on a sorted index": best_ms(lookup_by_index),
        "one isin + groupby": best_ms(lookup_all_at_once),
    },
    name="milliseconds",
).round(2)

same result: True True


200 boolean-mask scans                65.22
200 .loc lookups on a sorted index    39.46
one isin + groupby                     4.55
Name: milliseconds, dtype: float64

## 7. Build a frame once
`pd.concat` copies all the data it receives. Calling it inside a loop copies the growing frame again and again. Collect the pieces in a list and concatenate **once**.

In [11]:
pieces = [base.iloc[start : start + 20] for start in range(0, 4000, 20)]   # 200 small frames


def grow_in_loop():
    result = pieces[0]
    for piece in pieces[1:]:
        result = pd.concat([result, piece])
    return result


def concat_once():
    return pd.concat(pieces)


print("same result:", grow_in_loop().equals(concat_once()))
print(f"concat inside the loop: {best_ms(grow_in_loop):7.2f} ms")
print(f"one concat at the end : {best_ms(concat_once):7.2f} ms")

same result: True
concat inside the loop:  128.88 ms
one concat at the end :    7.43 ms


In [12]:
shutil.rmtree(workdir, ignore_errors=True)
print("temporary files removed:", not workdir.exists())

temporary files removed: True


## Tuning checklist
1. **Measure.** Find the slow step and the large columns before changing anything.
2. **Load less.** Only the columns and rows you need. Filter early.
3. **Right-size dtypes.** `category` for repeated text, smaller numbers where safe.
4. **Vectorise.** Replace row loops and `apply(axis=1)` (see notebook 07).
5. **Use built-in aggregations**, not lambdas.
6. **Avoid repeated work.** One merge or `groupby` instead of many lookups; one `concat` instead of many.
7. **Store intermediate data as Parquet.**
8. If the data still does not fit, process it in **chunks**, or push the heavy work to a database (Day 3).

## Key takeaways
- Timings on your machine will differ from the ones saved here. Compare approaches, not absolute numbers.
- Memory is dominated by text columns; `category` is the biggest single win when values repeat.
- Smaller numeric types trade precision or range for space.
- Do less work: load fewer columns, make fewer passes over the data, and check with a timer that each change helped.

## Practice
1. Convert `customers.csv` to efficient dtypes and report the memory saved.
2. Measure how long `big.sort_values("revenue")` takes, then `big.nlargest(10, "revenue")`.
3. Compute revenue per country from the CSV file in chunks of 25,000 rows.
4. Write `big` to Parquet without converting dtypes first. How does the file size compare with the `slim` version?